# Fan-out join · **01 — The RTT star schema**

I model the August 2026 incomplete pathways as a star: one fact table and
four dimensions. The grain is the whole argument of this piece, so I state
it before I build anything and then assert it.

**Fact grain: one row per period × provider × commissioner × treatment
function × week band, holding a count of incomplete pathways.**

A word on "pathway grain". NHS England doesn't publish one row per
pathway; the public extract is already aggregated. So in this repo
"correct grain" means that every pathway is counted in exactly one fact
row, and every sum over the fact table counts each pathway once. Any join
that preserves that is safe. Any join that repeats a fact row counts the
same pathways twice.

| table | key | what it holds |
|---|---|---|
| `fact_incomplete_pathways` | period, provider_code, commissioner_code, tf_code, week_band | pathways (count) |
| `dim_provider` | provider_code | name, host ICB (the extract's provider parent) |
| `dim_commissioner` | commissioner_code | name, parent ICB, or the hub itself for NHS England commissioning hubs |
| `dim_treatment_function` | tf_code | the 23 named treatment functions |
| `dim_week_band` | week_band | one-week bands 0-1 … 103-104, then 104+; within-18 and over-52 flags |

Every dimension joins to the fact **many-to-one**. That is the property the
fan-out in notebook 03 breaks.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, PERIOD, RTT_UNIDENTIFIED_COMMISSIONER, STANDARD_WEEKS,  # noqa: E402
                     STAR_DIR, TOTAL_TF_CODE, band_lower_week)

STAR_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)

## Step 1 — Read the extract

In [2]:
zip_path = BRONZE_DIR / "nhse_rtt_2026-08" / "Full-CSV-data-file-Aug26-ZIP-3M-vjgRdG.zip"
with zipfile.ZipFile(zip_path) as z:
    [name] = [n for n in z.namelist() if n.endswith(".csv")]
    with z.open(name) as f:
        raw = pd.read_csv(f, low_memory=False)
assert raw.Period.nunique() == 1
print(name, raw.shape, raw.Period.iloc[0])

bands = [c for c in raw.columns if c.startswith("Gt ")]
# 104 one-week bands plus the open-ended 104+ band, in order.
assert [band_lower_week(c) for c in bands] == list(range(105)), "band headers out of order"

inc = raw[raw["RTT Part Description"] == "Incomplete Pathways"].copy()
# The extract leaves empty bands blank rather than writing 0. A blank here
# means "no pathways in this band", so I make it an explicit 0 before any
# sum. (numpy sums propagate NaN; pandas sums skip it. I don't want the
# answer to depend on which one I happened to call.)
print(f"blank band cells filled with 0: {int(inc[bands].isna().sum().sum()):,}")
assert (inc[bands].fillna(0) >= 0).all().all() and (inc[bands].fillna(0) % 1 == 0).all().all()
inc[bands] = inc[bands].fillna(0).astype("int64")
# 'Total' is blank for incomplete pathways; 'Total All' is populated and must
# equal the band sum exactly, or the bands aren't the whole story.
inc["band_sum"] = inc[bands].sum(axis=1)
assert (inc.band_sum == inc["Total All"]).all(), "band sum != Total All"
print(f"incomplete pathway rows: {len(inc):,}")

20260831-RTT-August-2026-full-extract.csv (177770, 121) RTT-August-2026


blank band cells filled with 0: 857,857


incomplete pathway rows: 63,444


/tmp/ipykernel_883/1393161834.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  inc["band_sum"] = inc[bands].sum(axis=1)


## Step 2 — The two exclusions, both counted rather than lost

* `NONC` commissioner rows (no identified commissioner). NHS England's
  Table 1 and provider workbook both exclude them.
* The `C_999` all-specialty row. It's a subtotal, so it can't sit in a fact
  table beside the rows it sums. I check that the named functions add back
  up to it, provider × commissioner by provider × commissioner, before I drop it.

In [3]:
nonc = inc["Commissioner Org Code"].astype(str).str.startswith(RTT_UNIDENTIFIED_COMMISSIONER)
is_total = inc["Treatment Function Code"] == TOTAL_TF_CODE
print(f"NONC pathways dropped: {int(inc.loc[nonc & is_total, 'band_sum'].sum()):,}")

kept = inc[~nonc]
key = ["Provider Org Code", "Commissioner Org Code"]
by_tf = kept[kept["Treatment Function Code"] != TOTAL_TF_CODE].groupby(key)[bands].sum()
tot = kept[kept["Treatment Function Code"] == TOTAL_TF_CODE].set_index(key)[bands]
diff = (by_tf.sort_index() - tot.sort_index()).abs().to_numpy().max()
assert diff == 0, "named treatment functions do not sum to C_999"
print("named treatment functions sum exactly to C_999 for every provider x commissioner, every band")
kept = kept[kept["Treatment Function Code"] != TOTAL_TF_CODE]

NONC pathways dropped: 31,049


named treatment functions sum exactly to C_999 for every provider x commissioner, every band


## Step 3 — Dimensions, each with its key asserted unique

In [4]:
dim_provider = (kept[["Provider Org Code", "Provider Org Name", "Provider Parent Org Code", "Provider Parent Name"]]
                .drop_duplicates()
                .set_axis(["provider_code", "provider_name", "host_icb_code", "host_icb_name"], axis=1))
assert dim_provider.provider_code.is_unique, "a provider has more than one name or host ICB"

dim_commissioner = (kept[["Commissioner Org Code", "Commissioner Org Name", "Commissioner Parent Org Code", "Commissioner Parent Name"]]
                    .drop_duplicates()
                    .set_axis(["commissioner_code", "commissioner_name", "parent_code", "parent_name"], axis=1))
assert dim_commissioner.commissioner_code.is_unique
# NHS England's specialised commissioning hubs have no parent ICB in the
# extract. If I left the parent blank, any groupby on it would drop those
# pathways without a word (pandas drops NaN keys by default). So a hub is
# its own parent, and I flag it.
dim_commissioner["is_nhse_hub"] = dim_commissioner.parent_code.isna()
dim_commissioner["parent_code"] = dim_commissioner.parent_code.fillna(dim_commissioner.commissioner_code)
dim_commissioner["parent_name"] = dim_commissioner.parent_name.fillna(dim_commissioner.commissioner_name)

dim_treatment_function = (kept[["Treatment Function Code", "Treatment Function Name"]].drop_duplicates()
                          .set_axis(["tf_code", "tf_name"], axis=1).sort_values("tf_code", ignore_index=True))
assert dim_treatment_function.tf_code.is_unique and len(dim_treatment_function) == 23

dim_week_band = pd.DataFrame({"week_band": range(105)})
dim_week_band["label"] = [f">{w}-{w + 1}" if w < 104 else "104+" for w in dim_week_band.week_band]
# A band's upper bound is its lower week + 1. Within 18 weeks = upper <= 18,
# the definition NHS England, the verification pipeline and Two Maps all use.
dim_week_band["within_18"] = dim_week_band.week_band + 1 <= STANDARD_WEEKS
dim_week_band["over_52"] = dim_week_band.week_band >= 52
assert dim_week_band.within_18.sum() == 18

for n, d in [("dim_provider", dim_provider), ("dim_commissioner", dim_commissioner),
             ("dim_treatment_function", dim_treatment_function), ("dim_week_band", dim_week_band)]:
    print(f"{n:<24} {len(d):>4} rows")
print(f"commissioners that are NHS England hubs: {dim_commissioner.is_nhse_hub.sum()}")

dim_provider              532 rows
dim_commissioner          128 rows
dim_treatment_function     23 rows
dim_week_band             105 rows
commissioners that are NHS England hubs: 22


## Step 4 — The fact table

I unpivot the 105 band columns and drop zero cells: a fact row exists only
where there are pathways. That loses nothing, because a missing cell and a
zero cell sum the same.

In [5]:
wide = kept.rename(columns={"Provider Org Code": "provider_code", "Commissioner Org Code": "commissioner_code",
                            "Treatment Function Code": "tf_code"})
wide = wide[["provider_code", "commissioner_code", "tf_code", *bands]].rename(columns={c: band_lower_week(c) for c in bands})
fact = wide.melt(id_vars=["provider_code", "commissioner_code", "tf_code"], var_name="week_band", value_name="pathways")
fact = fact[fact.pathways > 0].astype({"week_band": "int16", "pathways": "int64"})
fact.insert(0, "period", PERIOD)
fact = fact.sort_values(["provider_code", "commissioner_code", "tf_code", "week_band"], ignore_index=True)

GRAIN = ["period", "provider_code", "commissioner_code", "tf_code", "week_band"]
assert not fact.duplicated(GRAIN).any(), "fact grain violated"
assert fact.pathways.sum() == kept[bands].to_numpy().sum(), "unpivot lost or invented pathways"
print(f"fact rows: {len(fact):,}   pathways: {fact.pathways.sum():,}")

fact rows: 501,422   pathways: 7,158,164


## Step 5 — Every dimension is many-to-one to the fact

For each dimension: every fact key finds exactly one dimension row. I
check it with `validate="many_to_one"`, which makes pandas raise if the
right-hand key isn't unique. That one argument is the whole difference
between this notebook and notebook 03.

In [6]:
checks = [(dim_provider, "provider_code"), (dim_commissioner, "commissioner_code"),
          (dim_treatment_function, "tf_code"), (dim_week_band, "week_band")]
for dim, k in checks:
    m = fact.merge(dim, on=k, how="left", validate="many_to_one", indicator=True)
    assert len(m) == len(fact) and (m._merge == "both").all(), k
    assert m.pathways.sum() == fact.pathways.sum()
print("all four joins: many-to-one, no orphans, row count and pathway total unchanged")

all four joins: many-to-one, no orphans, row count and pathway total unchanged


## Step 6 — Write the star

In [7]:
fact.to_parquet(STAR_DIR / "fact_incomplete_pathways.parquet", index=False)
for n, d in [("dim_provider", dim_provider), ("dim_commissioner", dim_commissioner),
             ("dim_treatment_function", dim_treatment_function), ("dim_week_band", dim_week_band)]:
    d.to_parquet(STAR_DIR / f"{n}.parquet", index=False)
print(sorted(p.name for p in STAR_DIR.iterdir()))

['dim_commissioner.parquet', 'dim_provider.parquet', 'dim_treatment_function.parquet', 'dim_week_band.parquet', 'fact_incomplete_pathways.parquet']
